<a href="https://colab.research.google.com/github/umaimafawad711-code/Loan_prediction/blob/main/week4/LoanPrediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Import Libraries


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder


%matplotlib inline

Load Data & Basic Exploration (EDA)

In [ ]:
df = pd.read_csv("dataset.csv")

print("Dataset Shape:", df.shape)
df.head()
df.info()
df.describe()

Bar Chart for Missing Values

In [ ]:

# Missing values
missing = df.isnull().sum()
missing[missing > 0].sort_values().plot(kind='bar')
plt.title("Missing Values Per Column")
plt.show()

print(df.isnull().sum())

Visual Representation Of Data

In [ ]:
# 1. Target Distribution
sns.countplot(x="Loan_Status", data=df)
plt.title("Loan Status Distribution (Y = Yes, N = No)")
plt.show()

# 2. Income Distribution (Histogram)
sns.histplot(df["ApplicantIncome"], bins=30, kde=True)
plt.title("Applicant Income Distribution")
plt.show()

# 3. Correlation Heatmap
numeric_df = df.select_dtypes(include=['int64', 'float64'])
sns.heatmap(numeric_df.corr(), annot=True, cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()

sns.countplot(x='Gender', hue='Loan_Status', data=df)

Drop Useless Columns


In [ ]:
# Loan_ID Drop
df = df.drop(columns=['Loan_ID'])

# Check missing values
print(df.isnull().sum())

# check Duplicates
print(df.duplicated().sum())
df = df.drop_duplicates()

Feature Engineering

In [ ]:
# Total Income
df['TotalIncome'] = df['ApplicantIncome'] + df['CoapplicantIncome']

# EMI (LoanAmount / Loan_Amount_Term)
df['EMI'] = df['LoanAmount'] / df['Loan_Amount_Term']

print(df[['ApplicantIncome', 'CoapplicantIncome', 'TotalIncome', 'LoanAmount', 'Loan_Amount_Term', 'EMI']].head())

In [ ]:
# X = Inputs
X = df.drop(columns=['Loan_Status'])

# y = Target
y = df['Loan_Status']

print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

In [ ]:
# Numeric or Categorical columns lists
numeric_features = ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount',
                    'Loan_Amount_Term', 'TotalIncome', 'EMI']
categorical_features = ['Gender', 'Married', 'Dependents', 'Education',
                        'Self_Employed', 'Credit_History', 'Property_Area']

# Pipeline for Numeric
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

#Pipeline for Categorical
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# combine Both
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

# Fit_transform
X_train_processed = preprocessor.fit_transform(X_train)

# Transform TEST only
X_test_processed = preprocessor.transform(X_test)

In [ ]:
print("X_train_processed shape:", X_train_processed.shape)
print("X_test_processed shape:", X_test_processed.shape)


# Check for missing values
print("Missing values in X_train:", np.isnan(X_train_processed).sum())
print("Missing values in X_test:", np.isnan(X_test_processed).sum())

**Linear Regression** was not used because our target variable (Loan_Status) is categorical (Y/N). Linear Regression is used for predicting continuous numerical values (like price), not categories. Logistic Regression is the correct equivalent for classification

KNN (K-Nearest Neighbors) Train


In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt

k_values = [1, 3, 5, 7, 9, 11, 13, 15]
accuracies = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_processed, y_train)
    y_pred_knn = knn.predict(X_test_processed)
    accuracies.append(accuracy_score(y_test, y_pred_knn))

plt.plot(k_values, accuracies, marker='o')
plt.title('K Value vs Accuracy')
plt.xlabel('K')
plt.ylabel('Accuracy')
plt.show()

Decision Tree Train

In [ ]:
from sklearn.tree import DecisionTreeClassifier

depths = [2, 3, 5, 10, None]
for d in depths:
    dt = DecisionTreeClassifier(max_depth=d, random_state=42)
    dt.fit(X_train_processed, y_train)
    y_pred_dt = dt.predict(X_test_processed)
    print(f"Max Depth: {d}, Accuracy: {accuracy_score(y_test, y_pred_dt)}")

Logistic Regression Train

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(random_state=42)
lr.fit(X_train_processed, y_train)

# 5 test rows ki probability dekho
probabilities = lr.predict_proba(X_test_processed[:5])
print("Probability of [Reject, Approve]:\n", probabilities)

Evaluate All Models

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
import pandas as pd

# ---------------------------------------------------------
# 1. KNN Model
# ---------------------------------------------------------
knn_best = KNeighborsClassifier(n_neighbors=5)
knn_best.fit(X_train_processed, y_train)
y_pred_knn = knn_best.predict(X_test_processed)

print("--- KNN (K=5) Classification Report ---")
print(classification_report(y_test, y_pred_knn))

# ---------------------------------------------------------
# 2. Decision Tree Model
# ---------------------------------------------------------
dt_best = DecisionTreeClassifier(max_depth=3, random_state=42)
dt_best.fit(X_train_processed, y_train)
y_pred_dt = dt_best.predict(X_test_processed)

print("\n--- Decision Tree (Depth=3) Classification Report ---")
print(classification_report(y_test, y_pred_dt))

# ---------------------------------------------------------
# 3. Logistic Regression Model
# ---------------------------------------------------------

lr_best = LogisticRegression(random_state=42)
lr_best.fit(X_train_processed, y_train)
y_pred_lr = lr_best.predict(X_test_processed)

print("\n--- Logistic Regression Classification Report ---")
print(classification_report(y_test, y_pred_lr))

# ---------------------------------------------------------
# 4. Table
# ---------------------------------------------------------
results = {
    'Model': ['KNN (K=5)', 'Decision Tree (Depth=3)', 'Logistic Regression'],
    'Accuracy': [
        accuracy_score(y_test, y_pred_knn),
        accuracy_score(y_test, y_pred_dt),
        accuracy_score(y_test, y_pred_lr)
    ],
    'Precision (Y)': [
        precision_score(y_test, y_pred_knn, pos_label='Y'),
        precision_score(y_test, y_pred_dt, pos_label='Y'),
        precision_score(y_test, y_pred_lr, pos_label='Y')
    ],
    'Recall (Y)': [
        recall_score(y_test, y_pred_knn, pos_label='Y'),
        recall_score(y_test, y_pred_dt, pos_label='Y'),
        recall_score(y_test, y_pred_lr, pos_label='Y')
    ],
    'F1-Score (Y)': [
        f1_score(y_test, y_pred_knn, pos_label='Y'),
        f1_score(y_test, y_pred_dt, pos_label='Y'),
        f1_score(y_test, y_pred_lr, pos_label='Y')
    ]
}

# Print table
df_results = pd.DataFrame(results)
print("\n--- FINAL COMPARISON TABLE ---")

print(df_results.round(3))

**Confusion Matrix Of Best Model**
Based on the results, Logistic Regression is chosen as the best model. Although Decision Tree had slightly higher accuracy (0.845), Logistic Regression provides probabilities which are highly useful for banking. Also, in a banking problem, a False Positive (approving a defaulter) is very dangerous, so we need a model with high Recall and F1-score, which Logistic Regression provides (Recall 0.98)

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns


y_pred_best_knn = lr_best.predict(X_test_processed)
print(classification_report(y_test, y_pred_best_knn))

# Confusion Matrix Heatmap
cm = confusion_matrix(y_test, y_pred_best_knn)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
import pandas as pd
import numpy as np

print("Training models with 5-Fold Cross-Validation...\n")

# 1. KNN Model
knn = KNeighborsClassifier(n_neighbors=5)
knn_scores = cross_val_score(knn, X_train_processed, y_train, cv=5, scoring='accuracy')

# 2. Decision Tree Model
dt = DecisionTreeClassifier(random_state=42)
dt_scores = cross_val_score(dt, X_train_processed, y_train, cv=5, scoring='accuracy')

# 3. Random Forest Model
rf = RandomForestClassifier(random_state=42)
rf_scores = cross_val_score(rf, X_train_processed, y_train, cv=5, scoring='accuracy')

# 4. Gradient Boosting Model
gb = GradientBoostingClassifier(random_state=42)
gb_scores = cross_val_score(gb, X_train_processed, y_train, cv=5, scoring='accuracy')

# 5. Logistic Regression Model (Added from Week 3)
lr = LogisticRegression(random_state=42, max_iter=1000) # max_iter increased to prevent warnings
lr_scores = cross_val_score(lr, X_train_processed, y_train, cv=5, scoring='accuracy')

# Create a table to show the results
results_cv = {
    'Model': ['KNN', 'Decision Tree', 'Random Forest', 'Gradient Boosting', 'Logistic Regression'],
    'Mean CV Score': [knn_scores.mean(), dt_scores.mean(), rf_scores.mean(), gb_scores.mean(), lr_scores.mean()],
    'Std Dev (Variance)': [knn_scores.std(), dt_scores.std(), rf_scores.std(), gb_scores.std(), lr_scores.std()]
}

df_cv_results = pd.DataFrame(results_cv)

# Sort the table by Mean CV Score (Highest first) to see the winner easily
df_cv_results = df_cv_results.sort_values(by='Mean CV Score', ascending=False)

print("--- 5-Fold Cross-Validation Results ---")
print(df_cv_results.round(4))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

# ==========================================
# 1. Tuning KNN (K Value vs CV Score)
# ==========================================
print("Tuning KNN...")
k_values = [1, 3, 5, 7, 9, 11, 13, 15]
knn_cv_scores = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    # We use cross_val_score on the training data only
    scores = cross_val_score(knn, X_train_processed, y_train, cv=5, scoring='accuracy')
    knn_cv_scores.append(scores.mean())

# Plotting KNN Results
plt.figure(figsize=(8, 4))
plt.plot(k_values, knn_cv_scores, marker='o', linestyle='dashed', color='blue')
plt.title('KNN: K Value vs Mean CV Accuracy')
plt.xlabel('K (Number of Neighbors)')
plt.ylabel('Mean CV Accuracy')
plt.xticks(k_values)
plt.grid(True)
plt.show()

# Find the best K
best_k = k_values[knn_cv_scores.index(max(knn_cv_scores))]
print(f"Best K for KNN: {best_k} with CV Score: {max(knn_cv_scores):.4f}\n")


# ==========================================
# 2. Tuning Decision Tree (Max Depth vs CV Score)
# ==========================================
print("Tuning Decision Tree...")
depths = [2, 3, 5, 10, None]
dt_cv_scores = []

for d in depths:
    dt = DecisionTreeClassifier(max_depth=d, random_state=42)
    scores = cross_val_score(dt, X_train_processed, y_train, cv=5, scoring='accuracy')
    dt_cv_scores.append(scores.mean())

# Plotting Decision Tree Results
# Note: We use str(d) for x-axis labels because 'None' cannot be plotted as a number
depth_labels = [str(d) for d in depths]

plt.figure(figsize=(8, 4))
plt.plot(depth_labels, dt_cv_scores, marker='s', linestyle='dashed', color='green')
plt.title('Decision Tree: Max Depth vs Mean CV Accuracy')
plt.xlabel('Max Depth (None = Unlimited)')
plt.ylabel('Mean CV Accuracy')
plt.grid(True)
plt.show()

# Find the best depth
best_depth = depths[dt_cv_scores.index(max(dt_cv_scores))]
print(f"Best Max Depth for Decision Tree: {best_depth} with CV Score: {max(dt_cv_scores):.4f}")

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

# Define the parameter grid
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, None]
}

# Create a base Random Forest model
rf_base = RandomForestClassifier(random_state=42)

# Setup GridSearchCV
# cv=5 means 5-fold cross-validation
grid_search = GridSearchCV(estimator=rf_base, param_grid=param_grid, cv=5, scoring='accuracy', n_jobs=-1)

# Fit the grid search to the training data
print("Running GridSearchCV for Random Forest (this might take a few seconds)...\n")
grid_search.fit(X_train_processed, y_train)

# Print the best parameters and best score
print("Best Parameters found by GridSearchCV:")
print(grid_search.best_params_)
print(f"\nBest Cross-Validation Score: {grid_search.best_score_:.4f}")

# Calculate how many trainings were done
# Total combinations = 3 (n_estimators) * 3 (max_depth) = 9 combinations
# Total fits = 9 combinations * 5 folds = 45 fits
total_fits = len(grid_search.cv_results_['params']) * 5
print(f"\nTotal trainings (fits) done: {total_fits}")

In [ ]:
from sklearn.metrics import accuracy_score
import pandas as pd

# We will train each model on the full training data and check train vs test score
# 1. KNN (Best K=9)
knn_tuned = KNeighborsClassifier(n_neighbors=9)
knn_tuned.fit(X_train_processed, y_train)
knn_train_acc = accuracy_score(y_train, knn_tuned.predict(X_train_processed))
knn_test_acc = accuracy_score(y_test, knn_tuned.predict(X_test_processed))

# 2. Decision Tree (Best Depth=2)
dt_tuned = DecisionTreeClassifier(max_depth=2, random_state=42)
dt_tuned.fit(X_train_processed, y_train)
dt_train_acc = accuracy_score(y_train, dt_tuned.predict(X_train_processed))
dt_test_acc = accuracy_score(y_test, dt_tuned.predict(X_test_processed))

# 3. Random Forest (Best Params from GridSearch)
rf_tuned = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf_tuned.fit(X_train_processed, y_train)
rf_train_acc = accuracy_score(y_train, rf_tuned.predict(X_train_processed))
rf_test_acc = accuracy_score(y_test, rf_tuned.predict(X_test_processed))

# 4. Logistic Regression (Our Week 3 Champion)
lr_final = LogisticRegression(random_state=42, max_iter=1000)
lr_final.fit(X_train_processed, y_train)
lr_train_acc = accuracy_score(y_train, lr_final.predict(X_train_processed))
lr_test_acc = accuracy_score(y_test, lr_final.predict(X_test_processed))

# Create the Overfitting Check Table
overfit_data = {
    'Model': ['KNN (K=9)', 'Decision Tree (Depth=2)', 'Random Forest (Tuned)', 'Logistic Regression'],
    'Train Score': [knn_train_acc, dt_train_acc, rf_train_acc, lr_train_acc],
    'Test Score': [knn_test_acc, dt_test_acc, rf_test_acc, lr_test_acc]
}

df_overfit = pd.DataFrame(overfit_data)
df_overfit['Gap'] = df_overfit['Train Score'] - df_overfit['Test Score']

# Sort by Gap (Lowest gap means most stable model)
df_overfit = df_overfit.sort_values(by='Gap', ascending=True)

print("--- Overfitting Check (Train vs Test) ---")
print(df_overfit.round(4))

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# We choose Logistic Regression as our final model based on its high Test Score (86.18%)
# and high explainability for banking.

print("--- Final Evaluation of Logistic Regression on Test Set ---\n")

# Make final predictions
final_predictions = lr_final.predict(X_test_processed)

# 1. Accuracy
print(f"Accuracy: {accuracy_score(y_test, final_predictions):.4f}\n")

# 2. Classification Report
print("Classification Report:")
print(classification_report(y_test, final_predictions))


In [ ]:
# Bonus: Gradient Boosting Tuning
gb_tuned = GradientBoostingClassifier(learning_rate=0.1, n_estimators=100, random_state=42)
gb_tuned.fit(X_train_processed, y_train)
gb_test_acc = accuracy_score(y_test, gb_tuned.predict(X_test_processed))
print(f"Tuned Gradient Boosting Test Accuracy: {gb_test_acc:.4f}")